# DINO / SAM + PaddleOCR Detection + Hezar CRNN

در Colab ابتدا T4 GPU را فعال کنید، سپس سلول‌ها را به ترتیب اجرا کنید. کد پروژه مستقیم از گیت دریافت می‌شود؛ آپلود ZIP و فایل اصلاحی لازم نیست.

Paddle Detection روی CPU و DINO/SAM و Hezar روی GPU اجرا می‌شوند. خروجی OCR پیشنهادی است و نیازمند بررسی صحت است.


## دریافت نسخهٔ یکپارچه
سلول بعد سورس یک commit مشخص و اصلاح‌شده را مستقیم دریافت می‌کند و به API گیت نیاز ندارد. شناسهٔ نسخه در خروجی ذخیره می‌شود.


In [ ]:
from google.colab import files
from pathlib import Path
import json
import subprocess
import sys
import tempfile
import zipfile
import urllib.request

REPOSITORY = 'neginnoori/Persian_OCR_IDCard'
BRANCH = 'feat/dino-sam-hezar-colab'
# Tested source bundle: direct commit archive; no GitHub API request.
SOURCE_COMMIT = 'f9d2923fabc16e64fb1071d1fb371522ba811f70'
SOURCE = Path(tempfile.mkdtemp(prefix='persian-id-source-', dir='/content'))
archive_path = SOURCE / 'source.zip'
url = f'https://codeload.github.com/{REPOSITORY}/zip/{SOURCE_COMMIT}'
with urllib.request.urlopen(url, timeout=120) as response:
    archive_path.write_bytes(response.read())
with zipfile.ZipFile(archive_path) as archive:
    if sum(m.file_size for m in archive.infolist()) > 100 * 1024 * 1024:
        raise ValueError('Source archive is unexpectedly large')
    for member in archive.infolist():
        target = (SOURCE / member.filename).resolve()
        if not target.is_relative_to(SOURCE.resolve()):
            raise ValueError('Invalid archive path')
        if (member.external_attr >> 16) & 0o170000 == 0o120000:
            raise ValueError('Source archive must not contain symlinks')
    archive.extractall(SOURCE)
projects = [p.parent for p in SOURCE.rglob('pyproject.toml')
            if (p.parent / 'scripts/setup_colab.py').is_file()
            and (p.parent / 'src/persian_id_ocr').is_dir()]
if len(projects) != 1:
    raise RuntimeError('Combined OCR project not found; upload the complete project ZIP')
PROJECT = projects[0]
print('Project:', PROJECT)
print('Source commit:', SOURCE_COMMIT)


## نصب و کنترل نسخه
این سلول کد محلی را حتی با شمارهٔ نسخهٔ یکسان دوباره نصب می‌کند و هش تمام ماژول‌های نصب‌شده را با سورس مقایسه می‌کند. پس از دریافت نسخهٔ تازه، این سلول را دوباره اجرا کنید. مدل‌ها در کش باقی می‌مانند.


In [ ]:
TEXT_DETECTOR = 'paddle'  # comparison option: craft
PADDLE_DEVICE = 'cpu'  # detection only; DINO/SAM/EasyOCR/Hezar remain on T4 GPU
subprocess.run([sys.executable, str(PROJECT / 'scripts/setup_colab.py'), '--paddle-device', PADDLE_DEVICE], check=True)
PYTHON = '/content/persian-id-venv/bin/python'
subprocess.run([PYTHON, '-u', str(PROJECT / 'scripts/verify_install.py'), str(PROJECT)], check=True)


## آپلود عکس‌های کارت
یک یا چند عکس انتخاب کنید. جهت به‌صورت خودکار تشخیص داده می‌شود. اگر استخراج نامطمئن باشد، هزار اجرا نمی‌شود و گزارش `needs_review` نمایش داده می‌شود.
برای بررسی دستی جهت می‌توانید `ORIENTATION_OVERRIDE` را تغییر دهید؛ این گزینه چرخش گوشه‌های کارت استخراج‌شده است، نه زاویهٔ عکس خام. هندسهٔ نامعتبر همچنان رد می‌شود.

In [ ]:
photos = files.upload()
INPUT_DIR = Path(tempfile.mkdtemp(prefix='persian-id-inputs-', dir='/content'))
INPUTS = []
for index, (name, content) in enumerate(photos.items()):
    destination = INPUT_DIR / f'{index:03d}_{Path(name).name}'
    destination.write_bytes(content)
    INPUTS.append(str(destination))
if not INPUTS:
    raise ValueError('Upload at least one card photo')
ORIENTATION_OVERRIDE = None  # None (خودکار) یا 0، 90، 180، 270


## اجرای استخراج و OCR
همهٔ عکس‌ها در یک process پردازش می‌شوند و مدل‌ها دوباره برای هر عکس بارگذاری نمی‌شوند. متن نهایی فقط از Hezar است؛ EasyOCR در مرحلهٔ تشخیص جهت استفاده می‌شود.

In [ ]:
subprocess.run([PYTHON, '-u', str(PROJECT / 'scripts/verify_install.py'), str(PROJECT)], check=True)
OUTPUT_ROOT = Path(tempfile.mkdtemp(prefix='persian-id-results-', dir='/content'))
command = [PYTHON, str(PROJECT / 'scripts/run_batch.py'), *INPUTS,
           '--output-root', str(OUTPUT_ROOT), '--cache-dir', '/content/persian-id-model-cache']
if ORIENTATION_OVERRIDE is not None:
    command += ['--orientation-override', str(ORIENTATION_OVERRIDE)]
command += ['--text-detector', TEXT_DETECTOR, '--paddle-device', PADDLE_DEVICE]
completed = subprocess.run(command)
runs = list(OUTPUT_ROOT.glob('run_*'))
if len(runs) != 1 or not (runs[0] / 'summary.json').is_file():
    raise RuntimeError('Run did not complete; inspect the error above')
RESULTS = runs[0]
SUMMARY = json.loads((RESULTS / 'summary.json').read_text())
(RESULTS / 'source_version.json').write_text(json.dumps({'repository': REPOSITORY, 'commit': SOURCE_COMMIT}))
print('Source commit:', SOURCE_COMMIT)
print(json.dumps(SUMMARY, ensure_ascii=False, indent=2))
if completed.returncode == 1:
    print('Runtime error in one or more images. Read the JSON reports below; OCR did not succeed for those images.')
elif completed.returncode == 2:
    print('Some images need review, contain no text, or have no detected card.')
elif completed.returncode != 0:
    raise RuntimeError(f'Unexpected process exit: {completed.returncode}')


In [ ]:
from IPython.display import display, Image as DisplayImage
for item in SUMMARY:
    folder = RESULTS / item['folder']
    report = json.loads((folder / 'result.json').read_text())
    print('\n', item['input'], '→', report['status'])
    for filename in ('card.png', 'card_needs_review.png', 'text_boxes.png', 'field_boxes.png'):
        if (folder / filename).is_file():
            print(filename)
            display(DisplayImage(filename=str(folder / filename)))
    print('متن خام / Raw text:')
    print(report.get('text', ''))
    if report.get('error'):
        print(report['error'])
    if report['status'] in ('needs_review', 'not_found'):
        print(json.dumps(report.get('metadata', {}).get('extraction', {}), ensure_ascii=False, indent=2))

    import pandas as pd
    display(pd.DataFrame(report.get('fields', [])))


## دانلود نتایج
ZIP شامل متن UTF-8، متن نرمال‌شده، تصویر کارت، کادرهای متن و گزارش JSON است. `ocr_candidate` به معنی صحت قطعی نیست. متن خام حفظ می‌شود و فیلدهای هویتی حدس زده نمی‌شوند.

In [ ]:
import shutil
result_zip = shutil.make_archive(str(RESULTS), 'zip', RESULTS)
files.download(result_zip)


## استفاده از تابع استخراج در پروژهٔ خودتان
پس از نصب هر دو پکیج در محیط برنامه:
```python
from cardid import CardExtractor
extractor = CardExtractor()  # یک نمونه برای هر process
card = extractor.extract("photo.jpg")  # تصویر PIL RGB، آماده برای OCR
```
برای استخراج و OCR با هم:
```python
from persian_id_ocr import PersianCardOCR
pipeline = PersianCardOCR(cache_dir="./models")
result = pipeline.process("photo.jpg")
print(result.status, result.text)
```
این مثال‌ها را داخل همان محیط مجازی اجرا کنید؛ پکیج‌ها در کرنل اصلی Colab نصب نشده‌اند. راهنمای بیشتر در README موجود است.